In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step6 DEG analysis
# 初始化环境
# ============================================================

import os
import pandas as pd
import numpy as np


project_dir = _hcc_path('')


step5_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step5_DEG"
)


result_dir = os.path.join(
    step5_dir,
    "DEG_Result"
)


fig_dir = os.path.join(
    step5_dir,
    "Figures"
)


os.makedirs(
    result_dir,
    exist_ok=True
)


os.makedirs(
    fig_dir,
    exist_ok=True
)


print(step5_dir)

In [ ]:
# ============================================================
# 读取counts矩阵
# ============================================================


count_file = os.path.join(
    step5_dir,
    "LIHC_Tumor_Normal_Counts.csv"
)


counts_gene = pd.read_csv(
    count_file,
    index_col=0
)


print(counts_gene.shape)


counts_gene.head()

In [ ]:
# ============================================================
# 创建metadata
# ============================================================


samples = counts_gene.columns


groups = []


for sample in samples:

    if "-11" in sample:
        groups.append("Normal")

    else:
        groups.append("Tumor")


metadata = pd.DataFrame(
    {
        "Sample": samples,
        "Group": groups
    }
)


metadata = metadata.set_index(
    "Sample"
)


print(
    metadata["Group"].value_counts()
)

In [ ]:
metadata.to_csv(
    os.path.join(
        result_dir,
        "Sample_Metadata.csv"
    )
)


print("metadata saved")

In [ ]:
# ============================================================
# 转置
# ============================================================


counts = counts_gene.T


print(
    counts.shape
)

In [ ]:
counts = counts.astype(int)


print(
    counts.dtypes.head()
)

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

print("PyDESeq2 ready")

In [ ]:
# ============================================================
# 创建DESeq2对象
# ============================================================

dds = DeseqDataSet(
    counts=counts,
    metadata=metadata,
    design="~ Group",
    refit_cooks=True
)


print("DDS created")

In [ ]:
# ============================================================
# Run DESeq2
# ============================================================

dds.deseq2()


print("DESeq2 finished")

In [ ]:
# ============================================================
# Tumor vs Normal
# ============================================================

stat_res = DeseqStats(
    dds,
    contrast=[
        "Group",
        "Tumor",
        "Normal"
    ]
)


stat_res.summary()

In [ ]:
# ============================================================
# DEG结果
# ============================================================

deg_result = stat_res.results_df.copy()


print(
    deg_result.shape
)


display(
    deg_result.head()
)

In [ ]:
deg_result.index.name = "Gene"


deg_result = (
    deg_result
    .reset_index()
)


display(
    deg_result.head()
)

In [ ]:
# ============================================================
# DEG分类
# ============================================================


deg_result["Significance"] = "NS"


deg_result.loc[
    (deg_result["padj"] < 0.05)
    &
    (deg_result["log2FoldChange"] > 1),
    "Significance"
] = "Up"


deg_result.loc[
    (deg_result["padj"] < 0.05)
    &
    (deg_result["log2FoldChange"] < -1),
    "Significance"
] = "Down"



print(
    deg_result["Significance"].value_counts()
)

In [ ]:
deg_result.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Tumor_vs_Normal_DEG_All.csv"
    ),
    index=False
)


print("DEG saved")

In [ ]:
# ============================================================
# Step5.14 Volcano plot 保存版
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import os


# 计算 -log10(padj)

deg_result["-log10_padj"] = (
    -np.log10(
        deg_result["padj"] + 1e-300
    )
)


plt.figure(
    figsize=(8,6)
)


for label in ["NS", "Up", "Down"]:

    temp = deg_result[
        deg_result["Significance"] == label
    ]

    plt.scatter(
        temp["log2FoldChange"],
        temp["-log10_padj"],
        s=8,
        label=label
    )


# 阈值线

plt.axvline(
    1,
    linestyle="--"
)

plt.axvline(
    -1,
    linestyle="--"
)

plt.axhline(
    -np.log10(0.05),
    linestyle="--"
)


plt.xlabel(
    "log2 Fold Change"
)


plt.ylabel(
    "-log10 adjusted P value"
)


plt.title(
    "LIHC Tumor vs Normal DEG Volcano Plot"
)


plt.legend()


# =============================
# 先保存
# =============================

volcano_png = os.path.join(
    fig_dir,
    "LIHC_DEG_Volcano.png"
)


plt.savefig(
    volcano_png,
    dpi=600,
    bbox_inches="tight"
)


print("Saved:")
print(volcano_png)


# 再显示

plt.show()

In [ ]:
# ============================================================
# Step5.15 Top DEG selection
# ============================================================

# 去除缺失padj
deg_heat = deg_result.dropna(
    subset=["padj"]
)


# Top50 Up
top_up = (
    deg_heat[
        deg_heat["Significance"]=="Up"
    ]
    .sort_values(
        by="log2FoldChange",
        ascending=False
    )
    .head(50)
)


# Top50 Down
top_down = (
    deg_heat[
        deg_heat["Significance"]=="Down"
    ]
    .sort_values(
        by="log2FoldChange",
        ascending=True
    )
    .head(50)
)


top_genes = list(
    top_up["Gene"]
) + list(
    top_down["Gene"]
)


print(
    "Heatmap genes:",
    len(top_genes)
)

In [ ]:
heatmap_data = counts_gene.loc[
    top_genes
]


print(
    heatmap_data.shape
)

In [ ]:
# ============================================================
# Step5.17 修正版 Z-score
# ============================================================

from scipy.stats import zscore


heatmap_z = pd.DataFrame(
    zscore(
        heatmap_data,
        axis=1
    ),
    index=heatmap_data.index,
    columns=heatmap_data.columns
)


print(
    heatmap_z.shape
)


heatmap_z.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(
    figsize=(12,8)
)


sns.heatmap(
    heatmap_z,
    cmap="RdBu_r",
    center=0,
    xticklabels=False,
    yticklabels=True
)


plt.title(
    "Top Differentially Expressed Genes"
)


plt.xlabel(
    "Samples"
)


plt.ylabel(
    "Genes"
)


plt.show()

In [ ]:
# ============================================================
# Heatmap sample ordering
# ============================================================


# 按 Normal → Tumor 排序

sample_order = (
    metadata
    .sort_values(
        by="Group"
    )
    .index
)


heatmap_z_ordered = heatmap_z.loc[
    :,
    sample_order
]


print(
    heatmap_z_ordered.shape
)

In [ ]:
# ============================================================
# Select representative samples
# ============================================================


normal_samples = (
    metadata[
        metadata["Group"]=="Normal"
    ]
    .sample(
        n=30,
        random_state=1
    )
    .index
)


tumor_samples = (
    metadata[
        metadata["Group"]=="Tumor"
    ]
    .sample(
        n=60,
        random_state=1
    )
    .index
)


selected_samples = list(
    normal_samples
) + list(
    tumor_samples
)


heatmap_plot = heatmap_z.loc[
    :,
    selected_samples
]


print(
    heatmap_plot.shape
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(
    figsize=(10,8)
)


sns.heatmap(
    heatmap_plot,
    cmap="RdBu_r",
    center=0,
    xticklabels=False,
    yticklabels=True
)


plt.title(
    "Top Differentially Expressed Genes\nTumor vs Normal"
)


plt.xlabel(
    "Samples"
)


plt.ylabel(
    "Genes"
)


plt.show()

In [ ]:
# ============================================================
# Step5.22 Top60 DEG heatmap
# ============================================================


top_up30 = (
    deg_result[
        deg_result["Significance"]=="Up"
    ]
    .sort_values(
        "log2FoldChange",
        ascending=False
    )
    .head(30)
)


top_down30 = (
    deg_result[
        deg_result["Significance"]=="Down"
    ]
    .sort_values(
        "log2FoldChange",
        ascending=True
    )
    .head(30)
)


heat_genes = list(
    top_up30["Gene"]
) + list(
    top_down30["Gene"]
)


print(len(heat_genes))

In [ ]:
# ============================================================
# Sample order
# ============================================================


sample_order = (
    metadata
    .sort_values(
        "Group"
    )
    .index
)


heatmap_data2 = heatmap_z.loc[
    heat_genes,
    sample_order
]


print(
    heatmap_data2.shape
)

In [ ]:
# ============================================================
# Step5.24 修改版
# DEG Heatmap + Tumor/Normal annotation
# ============================================================


import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os


# -----------------------------
# 1. 构建sample annotation
# -----------------------------


sample_group = metadata.loc[
    heatmap_data2.columns,
    "Group"
]


# 颜色映射
# Normal / Tumor

group_colors = {
    "Normal": "skyblue",
    "Tumor": "salmon"
}


col_colors = sample_group.map(
    group_colors
)


# -----------------------------
# 2. 绘制聚类热图
# -----------------------------


g = sns.clustermap(
    heatmap_data2,

    # 颜色
    cmap="RdBu_r",
    center=0,

    # 聚类
    method="complete",
    metric="euclidean",

    # 图大小
    figsize=(12,10),

    # 顶部注释条
    col_colors=col_colors,


    # 标签
    xticklabels=False,
    yticklabels=True,


    # 聚类树
    row_cluster=True,
    col_cluster=True
)



# 标题

g.fig.suptitle(
    "Top Differentially Expressed Genes\nLIHC Tumor vs Normal",
    y=1.02,
    fontsize=16
)



# -----------------------------
# 3. 保存
# -----------------------------


heatmap_png = os.path.join(
    fig_dir,
    "LIHC_DEG_Top60_Heatmap_annotation.png"
)


g.savefig(
    heatmap_png,
    dpi=600,
    bbox_inches="tight"
)



heatmap_svg = os.path.join(
    fig_dir,
    "LIHC_DEG_Top60_Heatmap_annotation.svg"
)


g.savefig(
    heatmap_svg,
    bbox_inches="tight"
)



print("Saved:")
print(heatmap_png)



plt.show()